# CinderTrace — Sri Lanka VIIRS observation evidence pilot

This notebook starts the next scientific check. It uses **public NASA catalogue records already retrieved and embedded here** for six prespecified days. No input upload is needed. It downloads standard Collection 2 fire masks and their exact-acquisition geolocation companions, checks their grids, and records local fire-mask classes and input-quality screens.

**Earthdata login is required inside this notebook**, even if you are already signed in in your browser. Do not paste credentials into a cell or share them. Login is not persisted. If you get a provider access denial, the run stops further downloads, keeps verified cached files and exports an honest partial result.

The full six-day pilot contains **68 files, approximately 5 GiB** according to catalogue sizes; geolocation files account for most of this. Processed results are much smaller. Keep the science cache for retries. This is a small feasibility sample, not a replacement-model validation dataset. It does not reconstruct historical activity, restore zero months, enable gap filling or change the frontend.

## 1. Install the readers
Run this once in a fresh kernel. Existing project research needs no changes. If your notebook asks for a kernel restart after installation, restart and continue from cell 2.

In [86]:
%pip -q install numpy pandas pyarrow shapely h3 netCDF4 earthaccess


## 2. Choose persistent storage
In Colab, the default mounts your Drive and stores the large cache there so an interrupted runtime can resume. In VS Code it uses the project data folder; change `OUTPUT_ROOT` if desired. Set `USE_DRIVE=False` only if temporary Colab storage is acceptable. The final download contains processed evidence, download receipts and metadata, not the 5 GiB science cache.

In [90]:
from pathlib import Path
import sys,json,hashlib,zipfile,io,base64
IN_COLAB='google.colab' in sys.modules
USE_DRIVE=True
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT=Path('/content/drive/MyDrive/CinderTrace_Observation_Evidence_v1')
else:
    OUTPUT_ROOT=Path('data/atlas/observation_evidence_run_v1')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
CACHE=OUTPUT_ROOT/'science_cache';CACHE.mkdir(exist_ok=True)
EVIDENCE=OUTPUT_ROOT/'catalogue';EVIDENCE.mkdir(exist_ok=True)
RESULTS=OUTPUT_ROOT/'results';RESULTS.mkdir(exist_ok=True)
print('Workspace:',OUTPUT_ROOT.resolve())


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace: /content/drive/MyDrive/CinderTrace_Observation_Evidence_v1


## 3. Verify and load the frozen public records
The embedded records include complete catalogue response pages and their receipts. S-NPP has no listed regional granules on the two sampled outage dates. That absence is kept unavailable; it is not a zero-fire observation. Later listed products remain recovery candidates until their science files are checked.

In [91]:
bundle_bytes=base64.b64decode('...')
assert hashlib.sha256(bundle_bytes).hexdigest()=='65a2956c6cefa24cf8c5a44d6f6571c476d7b1c1f11b2fc3b8e4561d48bc86e7'
with zipfile.ZipFile(io.BytesIO(bundle_bytes)) as archive:
    names=archive.namelist()
    assert len(names)==len(set(names))
    receipts=json.loads(archive.read('metadata_bundle_manifest.json'))
    assert set(names)==set(receipts)|{'metadata_bundle_manifest.json'}
    for name,receipt in receipts.items():
        path=Path(name)
        assert not path.is_absolute() and '..' not in path.parts
        raw=archive.read(name)
        assert len(raw)==receipt['bytes'] and hashlib.sha256(raw).hexdigest()==receipt['sha256'],name
        target=EVIDENCE/path;target.parent.mkdir(parents=True,exist_ok=True);target.write_bytes(raw)
    (EVIDENCE/'metadata_bundle_manifest.json').write_bytes(archive.read('metadata_bundle_manifest.json'))
INVENTORY=json.loads((EVIDENCE/'inventory.json').read_text())
import pandas as pd
from IPython.display import display
display(pd.DataFrame(INVENTORY['daily_catalogue']))
print('Science files:',len(INVENTORY['records']),'Catalogue MiB:',round(INVENTORY['catalogue_size_mb'],1))
print('Unique same-platform companions:',sum(r['status']=='unique_catalogue_companion' for r in INVENTORY['companions']))


,date,role,product,records,status
0,2021-08-15,recorded_zero_control_not_no_fire_truth,VNP14IMG,4,catalogue_listed
1,2021-08-15,recorded_zero_control_not_no_fire_truth,VJ114IMG,2,catalogue_listed
2,2021-08-15,recorded_zero_control_not_no_fire_truth,VNP03IMG,4,catalogue_listed
3,2021-08-15,recorded_zero_control_not_no_fire_truth,VJ103IMG,2,catalogue_listed
4,2021-08-19,positive_control,VNP14IMG,3,catalogue_listed
5,2021-08-19,positive_control,VJ114IMG,3,catalogue_listed
6,2021-08-19,positive_control,VNP03IMG,3,catalogue_listed
7,2021-08-19,positive_control,VJ103IMG,3,catalogue_listed
8,2022-08-04,listed_SNPP_missing_standard_date,VNP14IMG,0,no_catalogue_records_not_zero_observations
9,2022-08-04,listed_SNPP_missing_standard_date,VJ114IMG,5,catalogue_listed


Science files: 68 Catalogue MiB: 5071.6
Unique same-platform companions: 34


## 4. Load the screening method
The standard fire mask distinguishes cloud, water, land, unprocessed and three fire-confidence classes. All local classes are retained. The diagnostic clear-centre screen requires land/fire classification, nominal input-quality bits and no residual bowtie flag. Native fire classes have no vegetation attribution here; they are **not** the vegetation-only target from the prior transfer experiment.

The reader requires exact acquisition/platform/time companion matches and identical array shapes. Where sparse fires exist, it checks their zero-based line/sample coordinates against the geolocation grid. Granules without sparse fires have a weaker alignment check and are explicitly labelled. Pair candidates use granule midpoint separations of 30/60/90 minutes. Coarse H3 cell-centre overlap is exploratory: it does not establish identical pixel footprints, simultaneous fire conditions or full regional area coverage.

In [92]:
method_namespace={'__file__':str((OUTPUT_ROOT/'pipeline'/'evidence.py').resolve()),'__name__':'cindertrace_evidence'}
METHOD_SOURCE='"""Small VIIRS standard-data observation evidence pilot; never fills archive zeros.\nPublic metadata discovery runs without credentials. Science screening requires\nEarthdata-authorized files and is a pixel-centre diagnostic, not area completeness.\n"""\nfrom pathlib import Path\nfrom datetime import datetime,timezone\nimport hashlib,json,re,time,urllib.request,urllib.parse,urllib.error\nimport numpy as np\nimport pandas as pd\n\nROOT=Path(__file__).resolve().parent.parent\nOUT=ROOT/\'data/atlas/observation_evidence_pilot_v1\'\nPRODUCTS={\n \'VNP14IMG\':{\'collection\':\'C2734202914-LPCLOUD\',\'version\':\'002\',\'satellite\':\'Suomi-NPP\',\'kind\':\'fire\'},\n \'VJ114IMG\':{\'collection\':\'C2734197957-LPCLOUD\',\'version\':\'002\',\'satellite\':\'NOAA-20\',\'kind\':\'fire\'},\n \'VNP03IMG\':{\'collection\':\'C2105092163-LAADS\',\'version\':\'2\',\'satellite\':\'Suomi-NPP\',\'kind\':\'geolocation\'},\n \'VJ103IMG\':{\'collection\':\'C2105086226-LAADS\',\'version\':\'2.1\',\'satellite\':\'NOAA-20\',\'kind\':\'geolocation\'}}\nDAYS={\'2021-08-15\':\'recorded_zero_control_not_no_fire_truth\',\n \'2021-08-19\':\'positive_control\', \'2022-08-04\':\'listed_SNPP_missing_standard_date\',\n \'2022-08-12\':\'post_interruption_recovery_candidate\',\n \'2024-05-31\':\'listed_SNPP_missing_standard_date\',\n \'2024-06-05\':\'post_interruption_recovery_candidate\'}\nBOUNDARY_SHA=\'838db6bf67cb4d21df585d32483c64c9d56556d4fc216564bf4aa982c659382e\'\nSOURCES={\n \'fire_guide\':\'https://ladsweb.modaps.eosdis.nasa.gov/archive/Document%20Archive/Science%20Data%20Product%20Documentation/VIIRS_C2_AF-375m_User_Guide_1.0.pdf\',\n \'snpp_geolocation\':\'https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VNP03IMG/\',\n \'noaa20_geolocation\':\'https://ladsweb.modaps.eosdis.nasa.gov/missions-and-measurements/products/VJ103IMG\',\n \'cmr\':\'https://cmr.earthdata.nasa.gov/search/site/docs/search/api.html\'}\nPOLICY={\'pilot_id\':\'lk_viirs_observation_evidence_v1\',\'days\':DAYS,\'products\':PRODUCTS,\'sources\':SOURCES,\n \'sampling\':\'purposive schema/outage feasibility sample, not independent model validation\',\n \'standard_only\':True,\'h3_resolution\':8,\'candidate_time_windows_minutes\':[30,60,90],\n \'candidate_pairing\':\'all cross-sensor granule candidates within time windows; not assumed independent or one-to-one\',\n \'support\':\'pixel centres only; no footprint areas, cloud-free area fractions or complete coverage inferred\',\n \'screen\':\'fire-mask classes 5/7/8/9, nominal input QA bits 0–6, residual bowtie bit 22 absent; not an official replacement algorithm\',\n \'scope\':\'all native fire-mask classes, not vegetation-only model target; type attribution not independently available\',\n \'zero_rule\':\'missing catalogue, missing companion, failed download or screen is unavailable, never an observed zero\',\n \'replacement_enabled\':False,\'early_history_bands_enabled\':False,\'coverage_verified\':False,\'frontend_changed\':False}\n\ndef sha(raw):return hashlib.sha256(raw).hexdigest()\ndef acquisition(name):\n    match=re.search(r\'\\.A(\\d{7})\\.(\\d{4})\\.\',name)\n    if not match:raise ValueError(\'Unrecognized acquisition filename\')\n    return \'.\'.join(match.groups())\n\ndef query(params,destination):\n    url=\'https://cmr.earthdata.nasa.gov/search/granules.json?\'+urllib.parse.urlencode(params)\n    receipt=destination.with_suffix(\'.receipt.json\')\n    if destination.exists() and receipt.exists():\n        raw=destination.read_bytes();r=json.loads(receipt.read_text())\n        if r[\'url\']!=url or r[\'sha256\']!=sha(raw):raise ValueError(\'Changed metadata cache\')\n        return json.loads(raw),r\n    for attempt in range(4):\n        try:\n            with urllib.request.urlopen(urllib.request.Request(url,headers={\'User-Agent\':\'CinderTrace-observation-evidence/1.0\'}),timeout=60) as response:\n                raw=response.read();hits=response.headers.get(\'CMR-Hits\')\n            body=json.loads(raw)\n            if not isinstance(body.get(\'feed\',{}).get(\'entry\'),list):raise ValueError(\'Unexpected CMR schema\')\n            r={\'url\':url,\'sha256\':sha(raw),\'bytes\':len(raw),\'cmr_hits\':int(hits) if hits is not None else None,\'retrieved_utc\':datetime.now(timezone.utc).isoformat()}\n            destination.write_bytes(raw);receipt.write_text(json.dumps(r,indent=2)+\'\\n\')\n            return body,r\n        except (urllib.error.URLError,TimeoutError):\n            if attempt==3:raise\n            time.sleep(min(2**attempt,4))\n\ndef normalize(entry,product,day):\n    spec=PRODUCTS[product];name=entry[\'producer_granule_id\']\n    if entry.get(\'collection_concept_id\')!=spec[\'collection\'] or not name.startswith(product+\'.\'):raise ValueError(\'Wrong collection/product\')\n    urls=[l[\'href\'] for l in entry.get(\'links\',[]) if not l.get(\'inherited\') and l.get(\'rel\',\'\').endswith(\'/data#\') and l.get(\'href\',\'\').startswith(\'https://\') and urllib.parse.urlsplit(l[\'href\']).path.endswith(\'.nc\')]\n    if len(urls)!=1:raise ValueError(\'Expected one science NetCDF link\')\n    return {\'id\':entry[\'id\'],\'product\':product,\'satellite\':spec[\'satellite\'],\'kind\':spec[\'kind\'],\n     \'name\':name,\'acquisition\':acquisition(name),\'date\':day,\'time_start\':entry[\'time_start\'],\'time_end\':entry[\'time_end\'],\n     \'day_night\':entry.get(\'day_night_flag\',\'UNKNOWN\'),\'download_url\':urls[0],\n     \'catalogue_size_mb\':float(entry.get(\'granule_size\',0)),\'metadata_entry_sha256\':sha(json.dumps(entry,sort_keys=True).encode()),\n     \'catalogue_footprint_only\':True}\n\ndef join_companions(records):\n    fire=[r for r in records if r[\'kind\']==\'fire\'];result=[]\n    for f in fire:\n        g=[g for g in records if g[\'kind\']==\'geolocation\' and g[\'satellite\']==f[\'satellite\'] and g[\'acquisition\']==f[\'acquisition\']]\n        status=\'unique_catalogue_companion\' if len(g)==1 else \'missing_companion\' if not g else \'ambiguous_companion\'\n        if len(g)==1 and (pd.Timestamp(g[0][\'time_start\'])!=pd.Timestamp(f[\'time_start\']) or pd.Timestamp(g[0][\'time_end\'])!=pd.Timestamp(f[\'time_end\'])):status=\'time_extent_mismatch\'\n        result.append({\'fire_id\':f[\'id\'],\'geo_id\':g[0][\'id\'] if len(g)==1 else None,\'satellite\':f[\'satellite\'],\'date\':f[\'date\'],\'acquisition\':f[\'acquisition\'],\'status\':status})\n    return result\n\ndef time_candidates(records):\n    fire=[r for r in records if r[\'kind\']==\'fire\'];rows=[]\n    for a in fire:\n        if a[\'satellite\']!=\'Suomi-NPP\':continue\n        for b in fire:\n            if b[\'satellite\']!=\'NOAA-20\' or a[\'date\']!=b[\'date\']:continue\n            # Use separation of granule midpoints. Date-boundary candidates are out of this small pilot.\n            am=pd.Timestamp(a[\'time_start\'])+(pd.Timestamp(a[\'time_end\'])-pd.Timestamp(a[\'time_start\']))/2\n            bm=pd.Timestamp(b[\'time_start\'])+(pd.Timestamp(b[\'time_end\'])-pd.Timestamp(b[\'time_start\']))/2\n            dt=abs((am-bm).total_seconds())/60\n            if dt<=90:rows.append({\'snpp_id\':a[\'id\'],\'noaa20_id\':b[\'id\'],\'date\':a[\'date\'],\'midpoint_separation_minutes\':dt,\n             \'within_30\':dt<=30,\'within_60\':dt<=60,\'within_90\':True,\'day_night_agrees\':a[\'day_night\']==b[\'day_night\'] and a[\'day_night\'] in [\'DAY\',\'NIGHT\'],\n             \'status\':\'time_candidate_only_pixel_overlap_unverified\'})\n    return rows\n\ndef discover(boundary_path=None,output=None):\n    out=Path(OUT if output is None else output);out.mkdir(parents=True,exist_ok=True);rawdir=out/\'cmr\';rawdir.mkdir(exist_ok=True)\n    bp=Path(boundary_path or ROOT/\'data/atlas/canonical_lk_2000_2024_v1/region.geojson\');raw=bp.read_bytes()\n    if sha(raw)!=BOUNDARY_SHA:raise ValueError(\'Boundary differs from verified study region\')\n    from shapely.geometry import shape\n    region=shape(json.loads(raw)[\'features\'][0][\'geometry\']);bbox=\',\'.join(map(str,region.bounds));(out/\'region.geojson\').write_bytes(raw)\n    records=[];counts=[]\n    for day in DAYS:\n        for product,spec in PRODUCTS.items():\n            found=[];page=1;expected=None\n            while True:\n                params={\'collection_concept_id\':spec[\'collection\'],\'bounding_box\':bbox,\'temporal\':day+\'T00:00:00Z,\'+day+\'T23:59:59.999Z\',\'page_size\':200,\'page_num\':page}\n                body,receipt=query(params,rawdir/f\'{product}_{day}_{page}.json\');entries=body[\'feed\'][\'entry\']\n                if expected is None:expected=receipt[\'cmr_hits\']\n                elif receipt[\'cmr_hits\']!=expected:raise ValueError(\'Catalogue changed during pagination; retry in a fresh output folder\')\n                found.extend(entries)\n                if len(entries)<200:break\n                page+=1\n                if page>100:raise ValueError(\'Unexpected pilot catalogue size\')\n            if expected is not None and len(found)!=expected:raise ValueError(\'Incomplete CMR pagination\')\n            if len({e[\'id\'] for e in found})!=len(found):raise ValueError(\'Duplicate CMR entries\')\n            # Temporal queries overlap intervals; retain granules whose start falls on the requested UTC day.\n            found=[e for e in found if e[\'time_start\'][:10]==day]\n            normalized=[normalize(e,product,day) for e in found];records.extend(normalized)\n            counts.append({\'date\':day,\'role\':DAYS[day],\'product\':product,\'records\':len(normalized),\'status\':\'catalogue_listed\' if normalized else \'no_catalogue_records_not_zero_observations\'})\n        print(\'Catalogued\',day,flush=True)\n    joins=join_companions(records);candidates=time_candidates(records)\n    report={\'policy\':POLICY,\'created_utc\':datetime.now(timezone.utc).isoformat(),\'boundary_sha256\':sha(raw),\'bbox\':list(region.bounds),\n     \'records\':records,\'daily_catalogue\':counts,\'companions\':joins,\'time_candidates\':candidates,\n     \'catalogue_size_mb\':sum(r[\'catalogue_size_mb\'] for r in records),\n     \'status\':\'metadata_only_science_screen_pending\',\'science_files_verified\':0,\'usable_coverage_verified\':False}\n    (out/\'inventory.json\').write_text(json.dumps(report,indent=2,allow_nan=False)+\'\\n\');(out/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    return report\n\nclass ScienceAccessRequired(RuntimeError):pass\n\ndef download(record,session,cache):\n    """Resumable per file; stores no auth headers, cookies, tokens or passwords."""\n    cache=Path(cache);cache.mkdir(parents=True,exist_ok=True)\n    filename=Path(urllib.parse.urlsplit(record[\'download_url\']).path).name\n    if filename!=record[\'name\']+\'.nc\' and filename!=record[\'name\']:raise ValueError(\'URL/name mismatch\')\n    target=cache/filename;rp=cache/(filename+\'.receipt.json\')\n    if target.exists() and rp.exists():\n        r=json.loads(rp.read_text());raw=target.read_bytes()\n        if r[\'id\']!=record[\'id\'] or r[\'sha256\']!=sha(raw) or r[\'bytes\']!=len(raw):raise ValueError(\'Changed verified science cache\')\n        return target,r\n    part=cache/(filename+\'.part\');reason=\'network\'\n    for attempt in range(4):\n        try:\n            with session.get(record[\'download_url\'],stream=True,timeout=(30,180)) as response:\n                if response.status_code!=200:\n                    reason=\'HTTP \'+str(response.status_code)\n                    if response.status_code in (401,403):break\n                    response.raise_for_status()\n                expected=int(response.headers.get(\'Content-Length\',0));digest=hashlib.sha256();n=0\n                with part.open(\'wb\') as handle:\n                    for chunk in response.iter_content(1024*1024):\n                        if chunk:handle.write(chunk);digest.update(chunk);n+=len(chunk)\n                if expected and n!=expected:raise OSError(\'Incomplete content length\')\n            with part.open(\'rb\') as f:signature=f.read(8)\n            if signature!=b\'\\x89HDF\\r\\n\\x1a\\n\':raise ValueError(\'Response is not NetCDF4/HDF5 science data\')\n            r={\'id\':record[\'id\'],\'product\':record[\'product\'],\'acquisition\':record[\'acquisition\'],\'bytes\':n,\'sha256\':digest.hexdigest(),\n             \'retrieved_utc\':datetime.now(timezone.utc).isoformat(),\'http_status\':200,\'status\':\'content_length_and_signature_verified_not_provider_checksum\'}\n            part.replace(target);rp.write_text(json.dumps(r,indent=2)+\'\\n\');return target,r\n        except Exception as error:\n            # Do not print request exceptions; they can contain provider credential redirects.\n            reason=type(error).__name__\n            if attempt<3:time.sleep(min(2**attempt,4))\n        finally:\n            if part.exists():part.unlink()\n    if reason in [\'HTTP 401\',\'HTTP 403\']:raise ScienceAccessRequired(f\'{filename}: {reason}. Check Earthdata authorization for this provider; no zero written.\')\n    raise RuntimeError(f\'{filename}: {reason}. No zero written. Verified prior files retained; review Earthdata/provider access.\') from None\n\ndef variables(dataset,prefix=\'\'):\n    result={prefix+name:var for name,var in dataset.variables.items()}\n    for name,group in dataset.groups.items():result.update(variables(group,prefix+name+\'/\'))\n    return result\n\ndef find_variable(dataset,name):\n    norm=lambda s:re.sub(\'[^a-z0-9]\',\'\',s.lower())\n    hits=[(path,var) for path,var in variables(dataset).items() if norm(path.split(\'/\')[-1])==norm(name)]\n    if len(hits)!=1:raise ValueError(\'Missing or ambiguous science variable: \'+name)\n    return hits[0]\n\ndef screen_arrays(mask,qa,latitude,longitude,region):\n    """Chunk kernel: preserve every local class; good centres are an explicit screen."""\n    from shapely import intersects_xy\n    m=np.ma.asarray(mask);q=np.ma.asarray(qa);lat=np.ma.asarray(latitude);lon=np.ma.asarray(longitude)\n    if len(m.shape)!=2 or len({m.shape,q.shape,lat.shape,lon.shape})!=1:raise ValueError(\'Fire/geolocation grid mismatch\')\n    lat=np.ma.filled(lat.astype(float),np.nan);lon=np.ma.filled(lon.astype(float),np.nan)\n    # Invalid/masked QA must never be converted to nominal (zero).\n    goodgeo=np.isfinite(lat)&np.isfinite(lon)&(abs(lat)<=90)&(abs(lon)<=180)\n    local=goodgeo&intersects_xy(region,lon,lat)\n    mv=np.ma.filled(m.astype(float),np.nan);qv=np.ma.filled(q.astype(np.uint32),np.uint32(2**32-1))\n    known=~np.ma.getmaskarray(m)&np.isfinite(mv)&np.isin(mv,np.arange(10))\n    if (local&~known).any():raise ValueError(\'Unknown or missing local fire-mask class\')\n    nominal=(qv&127)==0;not_duplicate=(qv&(1<<22))==0\n    clear=local&np.isin(mv,[5,7,8,9])&nominal&not_duplicate&~np.ma.getmaskarray(q)\n    fires=clear&np.isin(mv,[7,8,9])\n    return {\'local\':local,\'mask\':mv,\'qa\':qv,\'qa_valid\':~np.ma.getmaskarray(q),\'clear\':clear,\'fires\':fires,\'latitude\':lat,\'longitude\':lon}\n\ndef screen_file(fire_path,geo_path,record,region):\n    from netCDF4 import Dataset\n    import h3\n    rows=[];summary={\'fire_id\':record[\'id\'],\'product\':record[\'product\'],\'satellite\':record[\'satellite\'],\'date\':record[\'date\'],\n      \'time_start\':record[\'time_start\'],\'time_end\':record[\'time_end\'],\'day_night\':record[\'day_night\'],\n      \'local_geolocated_centres\':0,\'nominal_clear_centres\':0,\'screened_fire_centres\':0,\'local_residual_bowtie_centres\':0,\n      \'scope\':\'all native fire classes; no vegetation type attribution\',\'status\':\'pixel_centre_screen_only_not_complete_coverage\'}\n    with Dataset(fire_path) as fire,Dataset(geo_path) as geo:\n        fm,mask=find_variable(fire,\'fire mask\');fq,qa=find_variable(fire,\'algorithm QA\');gl,lat=find_variable(geo,\'latitude\');go,lon=find_variable(geo,\'longitude\')\n        if len(mask.shape)!=2 or mask.shape!=lat.shape or mask.shape!=lon.shape or mask.shape!=qa.shape:raise ValueError(\'Companion grid shape mismatch\')\n        summary.update({\'fire_mask_variable\':fm,\'qa_variable\':fq,\'latitude_variable\':gl,\'longitude_variable\':go,\'shape\':list(mask.shape)})\n        # Independent sparse-fire coordinates check the zero-based line/sample alignment.\n        _,pl=find_variable(fire,\'FP_line\');_,ps=find_variable(fire,\'FP_sample\');_,plat=find_variable(fire,\'FP_latitude\');_,plon=find_variable(fire,\'FP_longitude\')\n        alignment=[]\n        for r,c,a,b in zip(pl[:100],ps[:100],plat[:100],plon[:100]):\n            r=int(r);c=int(c)\n            if r<0 or c<0 or r>=mask.shape[0] or c>=mask.shape[1]:raise ValueError(\'Invalid zero-based sparse-fire grid index\')\n            x=float(lat[r,c]);y=float(lon[r,c]);delta=max(abs(x-float(a)),abs(((y-float(b)+180)%360)-180))\n            if not np.isfinite(delta) or delta>.01:raise ValueError(\'Sparse-fire geolocation does not align with companion grid\')\n            alignment.append(delta)\n        summary[\'alignment_check_pixels\']=len(alignment);summary[\'alignment_max_error_degrees\']=max(alignment) if alignment else None\n        summary[\'alignment_status\']=\'sparse_fire_coordinates_agree\' if alignment else \'no_sparse_fires_only_catalogue_time_and_shape_checks\'\n        for start in range(0,mask.shape[0],256):\n            sl=slice(start,min(start+256,mask.shape[0]));x=screen_arrays(mask[sl,:],qa[sl,:],lat[sl,:],lon[sl,:],region)\n            summary[\'local_geolocated_centres\']+=int(x[\'local\'].sum());summary[\'nominal_clear_centres\']+=int(x[\'clear\'].sum());summary[\'screened_fire_centres\']+=int(x[\'fires\'].sum())\n            summary[\'local_residual_bowtie_centres\']+=int((x[\'local\']&x[\'qa_valid\']&((x[\'qa\']&(1<<22))!=0)).sum())\n            for cls in range(10):summary[\'class_\'+str(cls)]=summary.get(\'class_\'+str(cls),0)+int((x[\'local\']&(x[\'mask\']==cls)).sum())\n            ix=x[\'local\'];la=x[\'latitude\'][ix];lo=x[\'longitude\'][ix]\n            if not len(la):continue\n            chunk=pd.DataFrame({\'cell\':[h3.latlng_to_cell(float(a),float(b),8) for a,b in zip(la,lo)],\n              \'local_centres\':1,\'clear_centres\':x[\'clear\'][ix].astype(int),\'fire_centres\':x[\'fires\'][ix].astype(int)})\n            rows.append(chunk.groupby(\'cell\',as_index=False).sum())\n    cells=pd.concat(rows).groupby(\'cell\',as_index=False).sum() if rows else pd.DataFrame(columns=[\'cell\',\'local_centres\',\'clear_centres\',\'fire_centres\'])\n    for name in [\'fire_id\',\'satellite\',\'date\',\'time_start\',\'time_end\']:cells[name]=summary[name]\n    # Presence of a centre does not imply coverage of the full H3 cell.\n    return summary,cells\n\ndef candidate_diagnostics(candidates,cells):\n    rows=[]\n    for c in candidates:\n        base=dict(c);a=cells[cells.fire_id.eq(c[\'snpp_id\'])];b=cells[cells.fire_id.eq(c[\'noaa20_id\'])]\n        if not len(a) or not len(b):\n            base.update({\'status\':\'one_or_both_screens_missing_or_no_regional_centres\',\'common_clear_centre_cells\':None,\n              \'snpp_active_cells_on_common_support\':None,\'noaa20_active_cells_on_common_support\':None});rows.append(base);continue\n        # Cell labels approximate spatial co-location; actual pixel footprints differ.\n        common=a[a.clear_centres.gt(0)].merge(b[b.clear_centres.gt(0)],on=\'cell\',suffixes=(\'_snpp\',\'_noaa20\'))\n        base.update({\'status\':\'coarse_cell_centre_overlap_diagnostic_not_matched_pixel_footprints\',\n         \'common_clear_centre_cells\':len(common),\n         \'snpp_active_cells_on_common_support\':int(common.fire_centres_snpp.gt(0).sum()) if len(common) else None,\n         \'noaa20_active_cells_on_common_support\':int(common.fire_centres_noaa20.gt(0).sum()) if len(common) else None})\n        rows.append(base)\n    return pd.DataFrame(rows)\n\ndef daily_screen_status(inventory,summaries):\n    rows=[]\n    for day in DAYS:\n        for satellite in [\'Suomi-NPP\',\'NOAA-20\']:\n            planned=[r for r in inventory[\'records\'] if r[\'kind\']==\'fire\' and r[\'date\']==day and r[\'satellite\']==satellite]\n            done=[r for r in summaries if r[\'date\']==day and r[\'satellite\']==satellite]\n            all_done=bool(planned) and len(done)==len(planned)\n            local=sum(r[\'local_geolocated_centres\'] for r in done)\n            supported=all_done and local>0\n            rows.append({\'date\':day,\'satellite\':satellite,\'catalogue_fire_granules\':len(planned),\'screened_granules\':len(done),\n                \'status\':\'catalogue_absent_not_zero_observations\' if not planned else \'incomplete_science_screen\' if not all_done else \'no_local_centres_not_zero_activity\' if not local else \'catalogued_granules_screened_coverage_unverified\',\n                \'screened_fire_centres\':sum(r[\'screened_fire_centres\'] for r in done) if supported else None,\n                \'nominal_clear_centres\':sum(r[\'nominal_clear_centres\'] for r in done) if supported else None,\n                \'observing_coverage_verified\':False})\n    return pd.DataFrame(rows)\n\ndef run_science(inventory,session,cache,output):\n    from shapely.geometry import shape\n    out=Path(output);out.mkdir(parents=True,exist_ok=True);source=Path(inventory);inv=json.loads(source.read_text());region_raw=(source.parent/\'region.geojson\').read_bytes()\n    if sha(region_raw)!=BOUNDARY_SHA:raise ValueError(\'Boundary changed\')\n    region=shape(json.loads(region_raw)[\'features\'][0][\'geometry\']);lookup={r[\'id\']:r for r in inv[\'records\']};summaries=[];cells=[];receipts=[];failures=[]\n    # Each completed swath is retained immediately, so later access/schema failures do not lose prior work.\n    access_stopped=False\n    for join in inv[\'companions\']:\n        if access_stopped:\n            failures.append({**join,\'status\':\'not_attempted_after_access_denial\'});continue\n        if join[\'status\']!=\'unique_catalogue_companion\':failures.append({**join,\'status\':\'unavailable_companion\'});continue\n        f=lookup[join[\'fire_id\']];g=lookup[join[\'geo_id\']]\n        try:\n            fp,fr=download(f,session,cache);gp,gr=download(g,session,cache);receipts.extend([fr,gr])\n            summary,cell=screen_file(fp,gp,f,region);summaries.append(summary);cells.append(cell)\n            pd.DataFrame(summaries).to_parquet(out/\'granule_screening.parquet\',index=False)\n            pd.concat(cells,ignore_index=True).to_parquet(out/\'cell_centre_support.parquet\',index=False)\n            print(\'Screened\',f[\'name\'],flush=True)\n        except Exception as error:\n            # Error text may include authenticated URLs from a third-party library; retain safe type only.\n            failures.append({**join,\'status\':\'unavailable_science_screen\',\'error_type\':type(error).__name__})\n            if isinstance(error,ScienceAccessRequired):access_stopped=True\n            print(\'Unavailable screen:\',f[\'name\'],type(error).__name__,flush=True)\n        finally:\n            (out/\'download_receipts.json\').write_text(json.dumps(receipts,indent=2)+\'\\n\');(out/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    combined=pd.concat(cells,ignore_index=True) if cells else pd.DataFrame(columns=[\'fire_id\',\'cell\',\'clear_centres\',\'fire_centres\'])\n    pd.DataFrame(summaries,columns=None if summaries else [\'fire_id\',\'satellite\',\'date\',\'status\']).to_parquet(out/\'granule_screening.parquet\',index=False)\n    combined.to_parquet(out/\'cell_centre_support.parquet\',index=False)\n    (out/\'download_receipts.json\').write_text(json.dumps(receipts,indent=2)+\'\\n\')\n    (out/\'failures.json\').write_text(json.dumps(failures,indent=2)+\'\\n\')\n    daily=daily_screen_status(inv,summaries)\n    daily.to_parquet(out/\'daily_screen_status.parquet\',index=False)\n    diagnostic=candidate_diagnostics(inv[\'time_candidates\'],combined);diagnostic.to_parquet(out/\'candidate_pair_diagnostics.parquet\',index=False)\n    decision={\'pilot_id\':POLICY[\'pilot_id\'],\'inventory_sha256\':sha(source.read_bytes()),\'planned_fire_granules\':len(inv[\'companions\']),\n     \'screened_fire_granules\':len(summaries),\'failed_or_unavailable\':len(failures),\'access_denial_stopped_downloads\':access_stopped,\'status\':\'exploratory_pixel_centre_evidence_only\',\n     \'zero_months_restored\':0,\'replacement_validated\':False,\'gap_fill_enabled\':False,\'coverage_verified\':False,\n     \'independent_uncertainty_validation_completed\':False,\'frontend_changed\':False,\n     \'limitations\':[\'Small purposive sample; no model tuning or validation scores\',\'Granule times, not exact per-pixel observation times\',\n       \'Coarse cell-centre overlap is not matched physical footprints or full regional area support\',\'Candidate pairs may share granules; do not sum pair counts\',\n       \'No vegetation type attribution: do not compare directly to vegetation-only transfer scores\',\'Missing files and failed screens remain unavailable\']}\n    (out/\'decision.json\').write_text(json.dumps(decision,indent=2)+\'\\n\');(out/\'method.json\').write_text(json.dumps(POLICY,indent=2)+\'\\n\')\n    decision[\'files\']={}\n    for path in sorted(out.iterdir()):\n        if path.is_file() and path.name not in [\'manifest.json\']:\n            raw=path.read_bytes();decision[\'files\'][path.name]={\'bytes\':len(raw),\'sha256\':sha(raw)}\n    (out/\'manifest.json\').write_text(json.dumps(decision,indent=2,allow_nan=False)+\'\\n\')\n    return decision\n\n'
exec(compile(METHOD_SOURCE,'frozen_observation_evidence.py','exec'),method_namespace)
(RESULTS/'research_implementation.py').write_text(METHOD_SOURCE)


24242

## 5. Sign in to Earthdata
Enter your own username/password only at the interactive prompts. The notebook does not persist the login. The science files come from LP DAAC and LAADS; a 401/403 may require authorizing that provider in your Earthdata account. Browser sign-in alone does not authenticate this Python session. Never share login prompts, tokens or credential-bearing URLs in screenshots.

In [93]:
import earthaccess
auth=earthaccess.login(strategy='interactive',persist=False)
assert auth.authenticated,'Earthdata authentication was not completed.'
session=earthaccess.get_requests_https_session()
print('Authenticated for this running kernel; no credentials saved by this notebook.')


Authenticated for this running kernel; no credentials saved by this notebook.


## 6. Download and screen the full pilot
Verified cached files are reused. Each completed swath saves a checkpoint; incomplete transfers are retried without writing a zero. Rerun the login and processing cells after an access interruption. All diagnostics are recomputed from the verified cache, so stale partial tables cannot masquerade as completed results. Allow several GiB of storage and time for the first run.

In [94]:
try:
    DECISION=method_namespace['run_science'](EVIDENCE/'inventory.json',session,CACHE,RESULTS)
finally:
    session.close()
print(json.dumps({k:v for k,v in DECISION.items() if k!='files'},indent=2))
if DECISION['access_denial_stopped_downloads']:
    print('Stopped after provider access denial. Prior verified files are cached. Review Earthdata provider authorization, then rerun cells 5 and 6.')


Screened VNP14IMG.A2021227.0830.002.2024074081032
Screened VNP14IMG.A2021227.1924.002.2024074081032
Screened VNP14IMG.A2021227.1930.002.2024074081029
Screened VNP14IMG.A2021227.2106.002.2024074081029
Screened VJ114IMG.A2021227.0742.002.2024081173830
Screened VJ114IMG.A2021227.2018.002.2024081173829
Screened VNP14IMG.A2021231.0718.002.2024074084526
Screened VNP14IMG.A2021231.0854.002.2024074084527
Screened VNP14IMG.A2021231.1954.002.2024074084534
Screened VJ114IMG.A2021231.0806.002.2024081184907
Screened VJ114IMG.A2021231.1900.002.2024081184905
Screened VJ114IMG.A2021231.2042.002.2024081184906
Screened VJ114IMG.A2022216.0700.002.2024089191959
Screened VJ114IMG.A2022216.0706.002.2024089192001
Screened VJ114IMG.A2022216.0842.002.2024089191957
Screened VJ114IMG.A2022216.1936.002.2024089191957
Screened VJ114IMG.A2022216.1942.002.2024089191959
Screened VNP14IMG.A2022224.0700.002.2024076201654
Screened VNP14IMG.A2022224.0706.002.2024076201655
Screened VNP14IMG.A2022224.0842.002.2024076201657


## 7. Inspect availability before counts
An absent or failed screen is unavailable. A recorded zero is shown only after all catalogued swaths for that date/platform have been screened and local geolocated centres exist. Even then it means zero under this explicit screen, not a no-fire truth or complete daily coverage. Candidate rows can share source granules: do not sum them or treat them as independent model samples.

In [95]:
display(pd.read_parquet(RESULTS/'daily_screen_status.parquet'))
display(pd.read_parquet(RESULTS/'granule_screening.parquet'))
display(pd.read_parquet(RESULTS/'candidate_pair_diagnostics.parquet'))
display(pd.DataFrame(json.loads((RESULTS/'failures.json').read_text())))


,date,satellite,catalogue_fire_granules,screened_granules,status,screened_fire_centres,nominal_clear_centres,observing_coverage_verified
0,2021-08-15,Suomi-NPP,4,4,catalogued_granules_screened_coverage_unverified,0.0,6268.0,False
1,2021-08-15,NOAA-20,2,2,catalogued_granules_screened_coverage_unverified,0.0,79537.0,False
2,2021-08-19,Suomi-NPP,3,3,catalogued_granules_screened_coverage_unverified,69.0,536694.0,False
3,2021-08-19,NOAA-20,3,3,catalogued_granules_screened_coverage_unverified,203.0,469684.0,False
4,2022-08-04,Suomi-NPP,0,0,catalogue_absent_not_zero_observations,NaN,NaN,False
5,2022-08-04,NOAA-20,5,5,catalogued_granules_screened_coverage_unverified,0.0,54427.0,False
6,2022-08-12,Suomi-NPP,4,4,catalogued_granules_screened_coverage_unverified,3.0,362365.0,False
7,2022-08-12,NOAA-20,2,2,catalogued_granules_screened_coverage_unverified,12.0,380106.0,False
8,2024-05-31,Suomi-NPP,0,0,catalogue_absent_not_zero_observations,NaN,NaN,False
9,2024-05-31,NOAA-20,3,3,catalogued_granules_screened_coverage_unverified,1.0,474599.0,False


,fire_id,product,satellite,date,time_start,time_end,day_night,local_geolocated_centres,nominal_clear_centres,screened_fire_centres,...,class_0,class_1,class_2,class_3,class_4,class_5,class_6,class_7,class_8,class_9
0,G2919329708-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-15,2021-08-15T08:30:00.000Z,2021-08-15T08:36:00.000Z,DAY,479516,2893,0,...,0,119816,0,243,356564,2893,0,0,0,0
1,G2919329808-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-15,2021-08-15T19:24:00.000Z,2021-08-15T19:30:00.000Z,NIGHT,149229,0,0,...,79,37335,0,0,111814,0,1,0,0,0
2,G2919328621-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-15,2021-08-15T19:30:00.000Z,2021-08-15T19:36:00.000Z,NIGHT,312216,3375,0,...,25,76710,0,131,231975,3375,0,0,0,0
3,G2919328440-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-15,2021-08-15T21:06:00.000Z,2021-08-15T21:12:00.000Z,NIGHT,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,G2915163968-LPCLOUD,VJ114IMG,NOAA-20,2021-08-15,2021-08-15T07:42:00.000Z,2021-08-15T07:48:00.000Z,DAY,373359,6605,0,...,0,0,12478,44,354232,6605,0,0,0,0
5,G2915164061-LPCLOUD,VJ114IMG,NOAA-20,2021-08-15,2021-08-15T20:18:00.000Z,2021-08-15T20:24:00.000Z,NIGHT,368439,72932,0,...,0,0,0,2242,293265,72932,0,0,0,0
6,G2919341305-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-19,2021-08-19T07:18:00.000Z,2021-08-19T07:24:00.000Z,DAY,437087,263194,68,...,0,109401,0,7939,56528,263126,0,1,79,13
7,G2919343885-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-19,2021-08-19T08:54:00.000Z,2021-08-19T09:00:00.000Z,DAY,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,G2919344144-LPCLOUD,VNP14IMG,Suomi-NPP,2021-08-19,2021-08-19T19:54:00.000Z,2021-08-19T20:00:00.000Z,NIGHT,390319,273500,1,...,706,0,0,8582,107531,273499,0,0,1,0
9,G2915174067-LPCLOUD,VJ114IMG,NOAA-20,2021-08-19,2021-08-19T08:06:00.000Z,2021-08-19T08:12:00.000Z,DAY,390294,330438,202,...,5,0,0,9997,49801,330270,1,25,195,0


,snpp_id,noaa20_id,date,midpoint_separation_minutes,within_30,within_60,within_90,day_night_agrees,status,common_clear_centre_cells,snpp_active_cells_on_common_support,noaa20_active_cells_on_common_support
0,G2919329708-LPCLOUD,G2915163968-LPCLOUD,2021-08-15,48.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,0.0,NaN,NaN
1,G2919329808-LPCLOUD,G2915164061-LPCLOUD,2021-08-15,54.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,0.0,NaN,NaN
2,G2919328621-LPCLOUD,G2915164061-LPCLOUD,2021-08-15,48.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,837.0,0.0,0.0
3,G2919328440-LPCLOUD,G2915164061-LPCLOUD,2021-08-15,48.0,False,True,True,True,one_or_both_screens_missing_or_no_regional_cen...,NaN,NaN,NaN
4,G2919341305-LPCLOUD,G2915174067-LPCLOUD,2021-08-19,48.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,68489.0,61.0,172.0
5,G2919343885-LPCLOUD,G2915174067-LPCLOUD,2021-08-19,48.0,False,True,True,True,one_or_both_screens_missing_or_no_regional_cen...,NaN,NaN,NaN
6,G2919344144-LPCLOUD,G2915172704-LPCLOUD,2021-08-19,54.0,False,True,True,True,one_or_both_screens_missing_or_no_regional_cen...,NaN,NaN,NaN
7,G2919344144-LPCLOUD,G2915172557-LPCLOUD,2021-08-19,48.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,33564.0,1.0,1.0
8,G2920857287-LPCLOUD,G2921978602-LPCLOUD,2022-08-12,54.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,1060.0,0.0,0.0
9,G2921589527-LPCLOUD,G2921978602-LPCLOUD,2022-08-12,48.0,False,True,True,True,coarse_cell_centre_overlap_diagnostic_not_matc...,19338.0,2.0,10.0


""


## 8. Download one review ZIP
Run this even if the science download was partial. It includes the frozen catalogue, original metadata receipts, method, processed screens, candidate diagnostics and file hashes. It excludes credentials and the large science cache. Save this notebook with outputs and add the downloaded ZIP to the project folder. The next review will assess actual schema/alignment, local clear support and recovery evidence before designing a larger independent replacement test.

In [96]:
from datetime import datetime,timezone
import importlib.metadata
versions={name:importlib.metadata.version(name) for name in ['numpy','pandas','pyarrow','shapely','h3','netCDF4','earthaccess']}
(OUTPUT_ROOT/'runtime_versions.json').write_text(json.dumps(versions,indent=2)+'\n')
archive_path=OUTPUT_ROOT/('cindertrace_atlas_lk_observation_evidence_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'.zip')
files={}
for folder in [EVIDENCE,RESULTS]:
    for path in folder.rglob('*'):
        if path.is_file():files[path.relative_to(OUTPUT_ROOT).as_posix()]=path.read_bytes()
files['runtime_versions.json']=(OUTPUT_ROOT/'runtime_versions.json').read_bytes()
manifest={name:{'bytes':len(raw),'sha256':hashlib.sha256(raw).hexdigest()} for name,raw in sorted(files.items())}
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for name,raw in sorted(files.items()):archive.writestr(name,raw)
    archive.writestr('bundle_manifest.json',json.dumps(manifest,indent=2)+'\n')
print('Saved:',archive_path.resolve())
print('SHA256:',hashlib.sha256(archive_path.read_bytes()).hexdigest())
if IN_COLAB:
    from google.colab import files
    files.download(str(archive_path))


Saved: /content/drive/MyDrive/CinderTrace_Observation_Evidence_v1/cindertrace_atlas_lk_observation_evidence_20261115T001225Z.zip
SHA256: e9ee5e9edf6790a31dfad88bd7e6edc7de9f681f51e76e27e3ae132699c7fe10


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>